# Week 2 Text Preprocess and Benchmark & Integration
ทำการคลีนข้อความและแบ่งประโยคออกเป็น 3 ส่วน โดยใช้ pythainlp ทำการนับ token
1. สั้น
2. กลาง
3. ยาว 

นำประโยคไปให้โมเดลเสียงและเปรีนบเทียบ latency จากนั้นสร้างฟังก์ชั่น api รอรับ keyword จากฟอนต์

In [8]:
!python --version

Python 3.12.9


## Library

In [15]:
import os
import pandas as pd
import pythainlp 

import gtts 

In [10]:
try:
    import torch
    print("มี torch อยู่ไม่มืดแร้ว" "version:", torch.__version__ if hasattr(torch, '__version__') else "ไม่พบเวอร์ชั่น" )
except ImportError:
    print("อ้ายบ่มีtorch")
    

มี torch อยู่ไม่มืดแร้วversion: 2.9.1


## Data

In [33]:
current_path = os.getcwd()
print("ตอนนี้อยู่ใน:" ,{current_path})

dialogue2 = "Dialogue2.csv"

def dialogue(path):
    df = pd.read_csv(path)
    return df

df = dialogue(dialogue2)
print("โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ")
df.head(3)


ตอนนี้อยู่ใน: {'/Users/bam/AIDA-chatbot/backend/voice/dialogue2'}
โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ


,ID,Category,Length_Type,Scenario,Keyword,Original Script,TTS Script,Emotion
0,G01,Greeting,Long,ทักทายครั้งแรกแบบ1,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีค่า ไอด้านะคะ ยินดีที่ได้รู้จักไอด้าเป็น...,"สวัสดีค่า, ไอด้านะคะ. ยินดีที่ได้รู้จัก, ไอด้า...",Happy
1,G02,Greeting,Long,ทักทายครั้งแรกแบบ2,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีนะ ยินดีที่ได้รู้จักค่ะ ก่อนอื่นขอแนะนำต...,"สวัสดีนะ, ยินดีที่ได้รู้จักค่ะ. ก่อนอื่นขอแนะน...",Talking
2,G03,Greeting,Short,ทักทายแบบที่3,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",Hello สวัสดีนะคะ ไอด้าเอง ยินดีช่วยเหลือค่ะ สง...,"เฮลโล, สวัสดีนะคะ. ไอด้าเอง, ยินดีช่วยเหลือค่ะ...",Happy


## Text Preprocessing using Pythainlp
1. Count each ID to defining a length type for each sentenses ()
2. To improve context before use voice model (Phoneme/Token Preparation)

explaination: ช่วยลดความลำเอียง (Bias) ของผู้วิจัย และทำให้งานวิจัยนี้สามารถทำซ้ำ (Reproduce)
โมเดลนี้เข้ามาช่วยแยกคำเผื่อที่จะจัดเรียงให้โมเดลเสียงอ่านเข้าใจประโยคหรือคำประสมได้มากขึ้น
1. Short: ทักทายทั่วไปพูดคุยทั่วไปที่ไม่ได้ลงรายละเอียดมาก Low Latency
2. Medium: แทนการตอบคำถามทั่วไปในชีวิตประจำวัน 
3. Long: แทนการอธิบายข้อมูลทางเทคนิคหรือวิชาการที่มีความซับซ้อน

![Response Times](JakobNielsen.png)

In [61]:
import pandas as pd
import re
from pythainlp import word_tokenize
from pythainlp.corpus import thai_words
from tabulate import tabulate
from pythainlp.util import dict_trie

# # 1. สร้างลิสต์คำศัพท์เฉพาะของคณะเรา
# custom_words = {
#     "บียู ครอคส์", "ไอด้า", "ปัญญาประดิษฐ์", "วิทยาการข้อมูล", 
#     "วิศวกรรมศาสตร์", "ดาต้า ไซน์", "เอไอ", "ไพธอน", "อาทิฟิเชียล"
# }

# # 2. รวมคำศัพท์มาตรฐาน + คำศัพท์เฉพาะของเราเข้าด้วยกัน
# all_words = set(thai_words()) | custom_words
# custom_dict = dict_trie(all_words)

def get_word_details(text):
    if pd.isna(text):
        return 0, ""
    clean_text = re.sub(r'[^0-9a-zA-Z\u0e00-\u0e7f\s]', '', str(text))
    tokens = word_tokenize(clean_text, engine="newmm")
    tokens = [t.strip() for t in tokens if t.strip()]
    return len(tokens), "|".join(tokens)
   
def classify_length(count):
    if count <= 15: return "Short"
    elif count <= 40: return "Medium"
    else: return "Long"

# --- 1. คำนวณค่า ---
df['word_count'], df['token_preview'] = zip(*df['TTS Script'].apply(get_word_details))
df['length_newmm'] = df['word_count'].apply(classify_length)

# --- 2. จัดระเบียบ DF สำหรับการใช้งานจริง ---
# เราจะเลือกเก็บคอลัมน์สำคัญ + เกณฑ์ใหม่ที่แม่นยำกว่า
final_df = df[['ID', 'Category', 'Scenario', 'Keyword', 'Original Script', 
               'TTS Script', 'Emotion', 'word_count', 'length_newmm', 'token_preview']]

# --- 3. บันทึกลงไฟล์ใหม่ ---
final_df.to_csv('processed_dialogue_v2.csv', index=False, encoding='utf-8-sig')

print("✅ บันทึกไฟล์สำเร็จ: processed_dialogue_v2.csv")
print("-" * 30)

# แสดงตัวอย่างตารางที่เราจะนำไปทำต่อ
print(tabulate(final_df[['ID', 'length_newmm', 'word_count', 'TTS Script']].head(), 
               headers='keys', tablefmt='fancy_grid', showindex=False))

✅ บันทึกไฟล์สำเร็จ: processed_dialogue_v2.csv
------------------------------
╒══════╤════════════════╤══════════════╤═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│ ID   │ length_newmm   │   word_count │ TTS Script                                                                                                                                                  │
╞══════╪════════════════╪══════════════╪═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│ G01  │ Medium         │           27 │ สวัสดีค่า, ไอด้านะคะ. ยินดีที่ได้รู้จัก, ไอด้าเป็นผู้ช่วยตอบคำถามสำหรับคณะวิศวกรรมศาสตร์, สาขาปัญญาประดิษฐ์และวิทยาการข้อมูลค่ะ. มีอะไรอยากสอบถามไหมคะ?                                 │
├──────┼────────────────┼──────────────┼───────────────────────────────────────────────────────────

## TTS
1. การวัดประสิทธิภาพของโมเดล
2. การวัดเวลาของระบบทั้งหมด

## Edge

In [66]:
pip install mutagen

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import os
import pandas as pd
import edge_tts
from mutagen.mp3 import MP3
from tabulate import tabulate 

df_v2 = pd.read_csv('processed_dialogue_v2.csv')

output_folder = "output_edge_tts"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

async def run_benchmark(dataframe, output_path):
    latencies = []
    
    print("กำลังทดสอบระบบเสียงไอด้าบน...")
    
    for index, row in dataframe.iterrows():
        # จับเวลาเฉพาะช่วงที่สร้างเสียง (Inference Time)
        start = time.perf_counter()
        
        communicate = edge_tts.Communicate(
            row['TTS Script'], 
            "th-TH-PremwadeeNeural", 
            rate="+9%", pitch="+10Hz"
        )
        await communicate.save(os.path.join(output_path, f"{row['ID']}_edge.mp3"))
        
        end = time.perf_counter()
        latencies.append(end - start)
        
    dataframe['Latency'] = latencies
    return dataframe


# --- 1. รันการทดสอบ (ใช้ฟังก์ชันเดิมที่เขียนไว้) ---
results_df = await run_benchmark(df_v2, output_folder)

# --- 2. สร้างตารางสรุปผลรวม (Summary Report) ตามกลุ่ม Short, Medium, Long ---
summary = results_df.groupby('length_newmm')['Latency'].agg(['mean', 'min', 'max', 'count']).reset_index()
summary.columns = ['Group', 'Avg Latency (s)', 'Min (s)', 'Max (s)', 'Sample Size']

# --- 3. แสดงผลตารางเปรียบเทียบ ID รายตัวบนหน้าจอ ---
print("\n📋 รายงานผล Latency แยกตาม ID:")
comparison_table = results_df[['ID', 'length_newmm', 'word_count', 'Latency']]
print(tabulate(comparison_table, headers='keys', tablefmt='fancy_grid', showindex=False))

# --- 4. แสดงตารางสรุปเกณฑ์ความยาวประโยค ---
print("\n📊 สรุปผลตามกลุ่มความยาว (Comparison Summary):")
print(tabulate(summary, headers='keys', tablefmt='fancy_grid', showindex=False))


final_export_df = results_df[['ID', 'Category', 'word_count', 'length_newmm', 'Latency']]
final_export_df.to_csv('final_benchmark_report.csv', index=False, encoding='utf-8-sig')

กำลังทดสอบระบบเสียงไอด้าบน...

📋 รายงานผล Latency แยกตาม ID:
╒══════╤════════════════╤══════════════╤═══════════╕
│ ID   │ length_newmm   │   word_count │   Latency │
╞══════╪════════════════╪══════════════╪═══════════╡
│ G01  │ Medium         │           27 │  7.13878  │
├──────┼────────────────┼──────────────┼───────────┤
│ G02  │ Medium         │           36 │  0.693014 │
├──────┼────────────────┼──────────────┼───────────┤
│ G03  │ Medium         │           17 │  2.69462  │
├──────┼────────────────┼──────────────┼───────────┤
│ B01  │ Short          │           15 │  2.25838  │
├──────┼────────────────┼──────────────┼───────────┤
│ B02  │ Short          │           13 │  0.728055 │
├──────┼────────────────┼──────────────┼───────────┤
│ B03  │ Medium         │           20 │  0.574728 │
├──────┼────────────────┼──────────────┼───────────┤
│ K01  │ Medium         │           26 │  2.02614  │
├──────┼────────────────┼──────────────┼───────────┤
│ K02  │ Medium         │           29